# 12 — Publish research software

Connect software metadata in CWL to CodeMeta and citation, and distinguish publication metadata from security evidence.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/12-publish-research-software/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/12-publish-research-software/lesson.md) · [Exercise](../modules/12-publish-research-software/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/12-publish-research-software"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Generate CodeMeta from the contract

The publication generator describes research software. It does not inventory image contents, publish a release or prove security. Generate into scratch space and inspect both the software source and target application metadata.


In [ ]:
(
set -euo pipefail
transpiler-mate cwl2codemeta --code-repository https://github.com/eoap/waterbodies-learning.git     --output codemeta.json "$WORKSPACE/reference/waterbodies.cwl#waterbodies"
cat codemeta.json
)


## Compare release identity with citation

CodeMeta propagates `s:softwareVersion` from CWL. `CITATION.cff` is an editorial source that must be synchronized deliberately. Verify the version and license, then compare authorship, title and repository identity.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
import json
import os
from pathlib import Path
import yaml
root = Path(os.environ['WORKSPACE'])
contract = yaml.safe_load((root / 'reference/waterbodies.cwl').read_text())
citation = yaml.safe_load((root / 'CITATION.cff').read_text())
metadata = json.loads(Path('codemeta.json').read_text())
application = metadata['targetProduct']
print('CWL version:', contract['s:softwareVersion'])
print('CodeMeta version:', application['softwareVersion'])
print('Citation version:', citation['version'])
print('CodeMeta license:', application['license'], 'Citation license:', citation['license'])
assert application['softwareVersion'] == contract['s:softwareVersion'] == citation['version']
print((root / 'CITATION.cff').read_text())
PYTHON
)


## Practice an editorial release update

Create a scratch citation for 2.0.0. It must remain separate from the current 1.0.0 contract and CodeMeta; identify all sources that a real reviewed release would update together. Nothing in this notebook publishes to a registry, archive or GitHub.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
import os
from pathlib import Path
import yaml
citation = yaml.safe_load((Path(os.environ['WORKSPACE']) / 'CITATION.cff').read_text())
citation['version'] = '2.0.0'
Path('CITATION-next.cff').write_text(yaml.safe_dump(citation, sort_keys=False))
print(Path('CITATION-next.cff').read_text())
PYTHON
)


## Record your findings

**Expected outcome:** Generated CodeMeta and current citation match the contract version; you can distinguish citation, publication metadata and SBOM evidence.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/12-publish-research-software/solution.md) after the activity. Return to [the lab index](README.md).
